In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import requests

In [2]:
# Gets list of crypto tickers from coingecko and formats them into a pandas dataframe
crypto_request = requests.get("https://api.coingecko.com/api/v3/coins/markets",
                 params={"vs_currency": "usd", "per_page": 250, "order": "market_cap_desc"})
crypto_dataframe = pd.DataFrame(crypto_request.json()).iloc[:30]
crypto_dataframe[["id", "symbol", "name", "current_price", "market_cap", "total_volume"]]

,id,symbol,name,current_price,market_cap,total_volume
0,bitcoin,btc,Bitcoin,76300.000000,1532487866759,2.948856e+10
1,ethereum,eth,Ethereum,2433.300000,296958433268,1.530863e+10
2,tether,usdt,Tether,0.999145,183286566468,5.947836e+10
3,binancecoin,bnb,BNB,722.350000,96186162014,9.686751e+08
4,ripple,xrp,XRP,1.300000,81471581705,3.621733e+09
5,usd-coin,usdc,USDC,0.999563,73595593954,1.848378e+10
6,solana,sol,Solana,99.970000,58688059665,3.568126e+09
7,tron,trx,TRON,0.334603,31773010158,3.410322e+08
8,figure-heloc,figr_heloc,Figure Heloc,1.030000,23268681307,9.916325e+07
9,zcash,zec,Zcash,1350.070000,22862319022,2.598158e+09


In [3]:
# Reformats the ticker symbols and reformats them into COIN-USD form to be read by yahoo finance
ticker_symbols = crypto_dataframe["symbol"].map(lambda x: np.char.upper(x) + "-USD").values.astype(str).tolist()
ticker_symbols

['BTC-USD',
 'ETH-USD',
 'USDT-USD',
 'BNB-USD',
 'XRP-USD',
 'USDC-USD',
 'SOL-USD',
 'TRX-USD',
 'FIGR_HELOC-USD',
 'ZEC-USD',
 'HYPE-USD',
 'DOGE-USD',
 'USDS-USD',
 'XMR-USD',
 'RAIN-USD',
 'WBT-USD',
 'LINK-USD',
 'LEO-USD',
 'ADA-USD',
 'XLM-USD',
 'USDE-USD',
 'DAI-USD',
 'BCH-USD',
 'USD1-USD',
 'UNI-USD',
 'LTC-USD',
 'CC-USD',
 'GRAM-USD',
 'NEAR-USD',
 'AVAX-USD']

In [4]:
# Enters each ticker into Yahoo finance to check if it is valid
def attemptDownloads(tickers):
    valid_tickers = []
    for ticker in tickers:
        dataframe = yf.download(tickers=ticker, period="max", interval="1d")
        if len(dataframe) > 0:
            valid_tickers.append(ticker)
    return valid_tickers
valid_tickers = attemptDownloads(ticker_symbols)
valid_tickers

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FIGR_HELOC-USD"}}}
$FIGR_HELOC-USD: possibly delisted; no timezone found
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['FIGR_HELOC-USD']: possibly delisted; no timezone found
[*********************100%***********************]  1 of 1 completed
[*********************100%****************

['BTC-USD',
 'ETH-USD',
 'USDT-USD',
 'BNB-USD',
 'XRP-USD',
 'USDC-USD',
 'SOL-USD',
 'TRX-USD',
 'ZEC-USD',
 'HYPE-USD',
 'DOGE-USD',
 'USDS-USD',
 'XMR-USD',
 'RAIN-USD',
 'WBT-USD',
 'LINK-USD',
 'LEO-USD',
 'ADA-USD',
 'XLM-USD',
 'USDE-USD',
 'DAI-USD',
 'BCH-USD',
 'USD1-USD',
 'UNI-USD',
 'LTC-USD',
 'GRAM-USD',
 'NEAR-USD',
 'AVAX-USD']

In [ ]:
# ticker_universe = ["BTC-USD", "ETH-USD", "LINK-USD", "SOL-USD", "AVAX-USD", "XRP-USD", "BNB-USD", "LTC-USD", "ADA-USD", "DOT-USD", "DOGE-USD", "MATIC-USD"]

# I decided to create a clean universe of 20-30 coins picked myself from coinmarketcap.com. The coins are meant to be reasonably liquid, non-stablecoin cryptos that have at least 6 years of history.

# New ticker universe created by looking at coinmarketcap.com and picking non-stable coins that have a sufficiently large market cap and daily volume (500M USD AND 50M USD)
# they must also be at least 6 years old (started before September 2020)
# This means that some coins may appear during the backtesting time period but I don't think it will be that problematic
ticker_universe = ["BTC-USD", "ETH-USD", "BNB-USD", "XRP-USD", "SOL-USD", "TRX-USD", "ZEC-USD", "DOGE-USD", "XMR-USD", "LINK-USD", "ADA-USD", "XLM-USD", "BCH-USD", "UNI-USD", "LTC-USD", "NEAR-USD", "AVAX-USD", "HBAR-USD", "SHIB-USD", "XAUt-USD", "PAXG-USD", "KCS-USD", "ALGO-USD", "DASH-USD", "WBTC-USD", "WETH-USD", "BTCB-USD"]

In [295]:
# Downloads the prices from yfinance
daily_prices = yf.download(tickers=ticker_universe, period="max", interval="1d")

[*********************100%***********************]  27 of 27 completed


In [296]:
# Calculates daily returns and daily volumes from daily prices
daily_returns = (daily_prices["Close"] / daily_prices["Close"].shift() - 1).iloc[1:].loc["2020-01-01":]
daily_volumes = daily_prices["Volume"].loc["2020-01-01":]

In [297]:
def zScore(x, window=120):
    return (x - x.rolling(window).mean()) / x.rolling(window).std()

In [298]:
def sharpeRatio(x):
    return x.mean() / x.std() * np.sqrt(252)

In [ ]:
# Calculates weights for momentum strategy
# rolling periods for signal calculations can be varied

# I adjusted the rolling parameters for this function to slow down the signal to reduce transaction costs
# After playing around with the parameters for a little bit, I found these values to work best for the time being
def momentumWeights(returns, returns_rolling=45, volatility_rolling=120):
    signal = (zScore(returns.shift(), window=120).rolling(returns_rolling).mean() / returns.shift().rolling(volatility_rolling).std())
    ranked = signal.rank(axis=1)
    demeaned = ranked.sub(ranked.mean(axis=1), axis=0)
    weights = demeaned.div(demeaned.abs().sum(axis=1), axis=0)
    return weights

In [371]:
# Reversal strategy (still a work in progress)
def reversalWeights(returns, volumes, returns_rolling=7, volatility_rolling=720):
    signal = (returns.shift().rolling(returns_rolling).mean() / (volumes / volumes.rolling(volatility_rolling).mean())).dropna()
    ranked = signal.rank(axis=1)
    demeaned = ranked.sub(ranked.mean(axis=1), axis=0)
    normalised = demeaned.div(demeaned.abs().sum(axis=1), axis=0)
    return -normalised

In [372]:
# Creating the returns for the momentum strategy
momentum_strategy = (momentumWeights(daily_returns) * daily_returns).sum(axis=1)
reversal_strategy = (reversalWeights(daily_returns, daily_volumes) * daily_returns).sum(axis=1)

In [ ]:
# Calculating turnover and hence net returns of (momentum) strategy.
# the constrained backtest has a sharpe of 0.26.
def calculateTurnOver(weights):
    return (weights - weights.shift()).abs().sum(axis=1)

def calculateNetReturns(daily_returns, weights):
    gross_returns = (weights * daily_returns).sum(axis=1)
    turnover = calculateTurnOver(weights)
    net_returns = gross_returns.subtract(turnover * 0.002, fill_value=0)
    return net_returns

net_momentum_returns = calculateNetReturns(daily_returns, momentumWeights(daily_returns))
sharpeRatio(net_momentum_returns)

np.float64(0.2655156408597148)

In [374]:
# Calculates important summary metrics for the strategy
def strategySummary(strategy, rolling_window=14, market_threshold=0.08):
    market_trend = daily_returns['BTC-USD'].shift().rolling(window=rolling_window).sum()

    bullish_trend = market_trend > market_threshold
    bearish_trend = market_trend < -market_threshold
    choppy_trend = (market_trend > -market_threshold) & (market_trend < market_threshold)

    print(f"Overall Sharpe Ratio: {sharpeRatio(strategy):.3f}")
    print("")
    print(f"Choppy market Sharpe: {sharpeRatio(strategy[choppy_trend]):.3f}")
    print(f"Bullish market Sharpe: {sharpeRatio(strategy[bullish_trend]):.3f}")
    print(f"Bearish market Sharpe: {sharpeRatio(strategy[bearish_trend]):.3f}")
    print()
    print(f"Hit Rate: {(strategy > 0).mean():.3f}")
    print(f"Avg Win: {strategy[strategy > 0].mean():.3f}")
    print(f"Avg Loss: {strategy[strategy < 0].mean():.3f}")
    print(f"Win/Loss Ratio: {-strategy[strategy > 0].mean() / strategy[strategy < 0].mean():.2f}")
    print("")
    print(f"Percentage of choppy days: {choppy_trend.mean():.2f}")

In [ ]:
# the constrained backtest has a sharpe ratio of 0.26.
strategySummary(net_momentum_returns)

Overall Sharpe Ratio: 0.266

Choppy market Sharpe: 0.321
Bullish market Sharpe: 0.918
Bearish market Sharpe: -0.987

Hit Rate: 0.446
Avg Win: 0.008
Avg Loss: -0.007
Win/Loss Ratio: 1.15

Percentage of choppy days: 0.60


In [ ]:
# The unconstrained backtest has a sharpe ratio of 0.83.
strategySummary(momentum_strategy)

Overall Sharpe Ratio: 0.836

Choppy market Sharpe: 1.015
Bullish market Sharpe: 1.363
Bearish market Sharpe: -0.489

Hit Rate: 0.471
Avg Win: 0.008
Avg Loss: -0.007
Win/Loss Ratio: 1.16

Percentage of choppy days: 0.60
